# 02 · Объединение и признаки

Ноутбук собирает из очищенных таблиц две аналитические таблицы разного зерна (L5: «одна строка = что?»):

- **часовая** `hourly.csv` — одна строка = один час на сетке станции; для H3 (суточный цикл) и H4 (CAMS и станция);
- **суточная** `daily.csv` — одна строка = одни местные сутки Алматы; для H1, H2, нормы ВОЗ и модели.

Входы: `pm25_clean.csv`, `weather_clean.csv`, `cams_clean.csv` из `data/processed/`. Выходы: `hourly.csv` и `daily.csv` там же.

Порядок: читаем и проверяем → выравниваем осадки → объединяем → признаки времени → запись `hourly.csv` → суточная таблица (часть 2). Запускать сверху вниз.

In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src import clean, collect, features

PM25_CLEAN_PATH = "../data/processed/pm25_clean.csv"
WEATHER_CLEAN_PATH = "../data/processed/weather_clean.csv"
CAMS_CLEAN_PATH = "../data/processed/cams_clean.csv"
HOURLY_PATH = "../data/processed/hourly.csv"
DAILY_PATH = "../data/processed/daily.csv"

WEATHER_COLUMNS = list(collect.WEATHER_COLUMNS.values())
CAMS_COLUMNS = list(collect.CAMS_COLUMNS.values())

## Шаг 1. Читаем очищенные таблицы и проверяем контракты

Сначала убеждаемся, что входы соответствуют контрактам этапа 2. Отдельно проверяем границы: погода должна начинаться не позже станции и заканчиваться минимум через час после её последней метки — это нужно для сдвига осадков на следующем шаге.

In [2]:
pm25 = clean.read_clean_csv(PM25_CLEAN_PATH)
weather = clean.read_clean_csv(WEATHER_CLEAN_PATH)
cams = clean.read_clean_csv(CAMS_CLEAN_PATH)

clean.check_clean_pm25(pm25)
clean.check_clean_hourly(weather, WEATHER_COLUMNS)
clean.check_clean_hourly(cams, CAMS_COLUMNS)

print("pm25:", pm25.shape, "| weather:", weather.shape, "| cams:", cams.shape)

# погода покрывает станцию и ещё час после её последней метки
assert weather["time_utc"].min() <= pm25["time_utc"].min()
assert weather["time_utc"].max() >= pm25["time_utc"].max() + pd.Timedelta(hours=1)

print("станция:", pm25["time_utc"].min(), "...", pm25["time_utc"].max())
print("погода: ", weather["time_utc"].min(), "...", weather["time_utc"].max())
print("CAMS начинается:", cams["time_utc"].min())

pm25: (49075, 4) | weather: (49104, 7) | cams: (28776, 2)
станция: 2020-04-09 16:00:00+00:00 ... 2025-11-14 10:00:00+00:00
погода:  2020-04-09 00:00:00+00:00 ... 2025-11-14 23:00:00+00:00
CAMS начинается: 2022-08-04 00:00:00+00:00


## Шаг 2. Осадки: тот же час, что у PM2.5

Метка времени у станции — начало часа: PM2.5 в строке T — среднее за [T, T+1 ч). У Open-Meteo `temperature_2m`, влажность, давление и ветер — мгновенные значения в момент T, а `precipitation` — «сумма за предыдущий час» (по документации Open-Meteo: *Preceding hour sum*). Значит, осадки в метке T относятся к часу [T−1 ч, T).

Чтобы осадки и PM2.5 в одной строке описывали один и тот же час, берём осадки из следующей строки (`shift(-1)`) — на полной часовой сетке это следующий час. Без сдвига суточная сумма попала бы на час мимо суток (решение — ADR-014).

In [3]:
weather_aligned = features.align_precip_to_hour_start(weather)

# первые часы рядом: до и после сдвига
compare = pd.DataFrame(
    {
        "time_utc": weather["time_utc"],
        "precip_before": weather["precip"],
        "precip_after": weather_aligned["precip"],
    }
)
print(compare[compare["precip_before"] > 0].head(5))

# сумма осадков на часах станции: сдвиг меняет общий объём только на краях периода
# (на один час); здесь крайние часы без осадков, поэтому суммы совпадают
station_hours = pm25["time_utc"]
before = weather[weather["time_utc"].isin(station_hours)]["precip"].sum()
after = weather_aligned[weather_aligned["time_utc"].isin(station_hours)]["precip"].sum()
print("сумма осадков на часах станции, до:", round(before, 1), "| после:", round(after, 1))
print("NaN в precip после сдвига (всего):", weather_aligned["precip"].isna().sum())

# суточные суммы по местным датам: считаем только сутки станции (те же, что попадут в daily.csv)
dates = features.local_dates(weather["time_utc"])
sum_before = weather.groupby(dates)["precip"].sum()
sum_after = weather_aligned.groupby(dates)["precip"].sum()
hours_in_day = weather.groupby(dates)["time_utc"].size()
station_hours_in_day = pm25.groupby(features.local_dates(pm25["time_utc"]))["time_utc"].size()
station_full_days = station_hours_in_day[station_hours_in_day >= 24].index
diff = (sum_after[station_full_days] - sum_before[station_full_days]).abs()
print("полных суток станции:", len(station_full_days))
print("из них с изменившейся суточной суммой:", (diff > 1e-9).sum())
print("максимум разницы суточной суммы, мм:", round(diff.max(), 1))

                   time_utc  precip_before  precip_after
0 2020-04-09 00:00:00+00:00            0.5           0.5
1 2020-04-09 01:00:00+00:00            0.5           0.3
2 2020-04-09 02:00:00+00:00            0.3           0.2
3 2020-04-09 03:00:00+00:00            0.2           0.4
4 2020-04-09 04:00:00+00:00            0.4           0.6
сумма осадков на часах станции, до: 2925.2 | после: 2925.2
NaN в precip после сдвига (всего): 1
полных суток станции: 2044
из них с изменившейся суточной суммой: 303
максимум разницы суточной суммы, мм: 7.4


## Шаг 3. Объединение по `time_utc`

Строки задаёт станция: `how="left"` от таблицы PM2.5, поэтому сетка станции не меняется (ADR-012). `validate="one_to_one"` — защита от тихого размножения строк, если в погоде или CAMS вдруг окажется повторяющийся час; после каждого `merge` функция проверяет число строк.

CAMS в Open-Meteo начинается с 2022-08-04 — раньше значений нет. Это период источника, а не пропуск, поэтому `cams_pm25` до этой даты остаётся пустым и мы его не заполняем.

In [4]:
merged = features.merge_sources(pm25, weather_aligned, cams)

print("строк: станция", len(pm25), "-> после объединения", len(merged))
print(merged.isna().sum())

# пустой CAMS ровно там, где час раньше первой метки источника
first_cams = cams["time_utc"].min()
hours_before_cams = (merged["time_utc"] < first_cams).sum()
print("часов станции раньше начала CAMS:", hours_before_cams)
assert merged["cams_pm25"].isna().sum() == hours_before_cams

строк: станция 49075 -> после объединения 49075
time_utc                 0
pm25                 10235
pm25_missing             0
pm25_interpolated        0
temp_c                   0
humidity                 0
precip                   0
pressure                 0
wind_speed               0
wind_dir                 0
cams_pm25            20312
dtype: int64
часов станции раньше начала CAMS: 20312


## Шаг 4. Признаки времени по местному времени

Ключ таблицы остаётся `time_utc`, а признаки считаем по местному времени Алматы: суточный цикл и сезон отопления определяются жизнью города, а не UTC. 18:00 UTC — это уже полночь в Алматы и следующий день. 1 марта 2024 в Алматы сменился пояс (UTC+6 → UTC+5), поэтому местные сутки 29.02.2024 длятся 25 часов.

Определения:
- `season` — метеорологические сезоны по месяцу: зима — дек–фев, весна — мар–май, лето — июн–авг, осень — сен–ноя;
- `heating_season` — с 15 октября по 15 апреля, обе границы включительно (ориентир; реальные даты каждый год свои);
- `is_weekend` — суббота и воскресенье;
- `is_calm` — штиль: ветер строго меньше 1 м/с.

In [5]:
hourly = features.make_hourly(merged)
print(hourly.head())

print(hourly["season"].value_counts())
print(hourly["heating_season"].value_counts())
print("доля часов со штилем:", round(hourly["is_calm"].mean(), 4))

# смена пояса: местный час 23 встречается дважды
around_change = hourly[
    (hourly["time_utc"] >= pd.Timestamp("2024-02-29 17:00", tz="UTC"))
    & (hourly["time_utc"] <= pd.Timestamp("2024-02-29 19:00", tz="UTC"))
]
print(around_change[["time_utc", "hour", "month"]])

                   time_utc  pm25  pm25_missing  pm25_interpolated  temp_c  \
0 2020-04-09 16:00:00+00:00   6.0         False              False    -2.4   
1 2020-04-09 17:00:00+00:00  10.0         False              False    -2.2   
2 2020-04-09 18:00:00+00:00   8.0         False              False    -4.4   
3 2020-04-09 19:00:00+00:00   6.0         False              False    -5.0   
4 2020-04-09 20:00:00+00:00   4.0         False              False    -5.9   

   humidity  precip  pressure  wind_speed  wind_dir  cams_pm25  hour  \
0        84     0.0    1039.4        1.88       115        NaN    22   
1        72     0.0    1039.0        1.70       130        NaN    23   
2        68     0.0    1038.7        2.14       127        NaN     0   
3        71     0.0    1038.0        2.06       129        NaN     1   
4        73     0.0    1037.5        1.98       131        NaN     2   

   dayofweek  month  is_weekend  season  heating_season  is_calm  
0          3      4       False

## Шаг 5. Проверки и запись `hourly.csv`

Контракт `check_hourly` вызываем до записи и после чтения. Формат времени тот же, что у очищенных файлов (`+0000`). Файл перезаписывается при каждом запуске. После чтения сравниваем таблицу с исходной: типы `hour`, `dayofweek` и `month` после чтения CSV становятся `int64` (до записи — `int32`), поэтому сравниваем значения без учёта типа.

In [6]:
features.check_hourly(hourly)
clean.save_clean_csv(hourly, HOURLY_PATH)

hourly_read = clean.read_clean_csv(HOURLY_PATH)
features.check_hourly(hourly_read)
pd.testing.assert_frame_equal(hourly, hourly_read, check_dtype=False)

print("hourly:", hourly.shape, "| после чтения:", hourly_read.shape)

with open(HOURLY_PATH, encoding="utf-8") as file:
    print(file.readline().strip())
    print(file.readline().strip())

hourly: (49075, 18) | после чтения: (49075, 18)
time_utc,pm25,pm25_missing,pm25_interpolated,temp_c,humidity,precip,pressure,wind_speed,wind_dir,cams_pm25,hour,dayofweek,month,is_weekend,season,heating_season,is_calm
2020-04-09T16:00:00+0000,6.0,False,False,-2.4,84,0.0,1039.4,1.88,115,,22,3,4,False,spring,True,False


## Шаг 6. Местные сутки и полнота дня

Суточная таблица: одна строка = одни местные сутки Алматы. Берём только **полные** сутки — на краях периода станции сутки неполные (в первый день станция включилась вечером, в последний — данные кончаются днём), и среднее по ним описывало бы часть дня. Из-за смены пояса 1 марта 2024 местные сутки 29.02.2024 длятся 25 часов — все они попадают в одну дату.

Суточное среднее PM2.5 считаем, только если в сутках есть не меньше 18 часов со значением (порог абсолютный, и для 25-часовых суток тоже). Интерполированные часы считаем: это дыры не длиннее 3 часов между двумя измерениями, мы сами решили им доверять на этапе очистки (ADR-013). Чтобы видеть цену этого решения, ниже сравниваем с вариантом «только измеренные часы».

In [7]:
daily_raw = features.aggregate_daily(hourly)

# сколько часов во всех местных датах до фильтра (видны неполные края и 25-часовые сутки)
hours_per_date = features.local_dates(hourly["time_utc"]).value_counts()
print("часов в местных сутках -> число дат:")
print(hours_per_date.value_counts().sort_index())

print(daily_raw[daily_raw["date"] == "2024-02-29"])
print("полных суток:", len(daily_raw))
print("первая дата:", daily_raw["date"].min().date(), "| последняя:", daily_raw["date"].max().date())

# распределение числа часов со значением PM2.5
bins = pd.cut(
    daily_raw["pm25_hours"],
    bins=[-1, 0, 17, 23, 24, 25],
    labels=["0", "1-17", "18-23", "24", "25"],
)
print(bins.value_counts().sort_index())

with_mean = daily_raw["pm25_mean"].notna().sum()
print("суток с pm25_mean (>= 18 часов):", with_mean, "| без:", daily_raw["pm25_mean"].isna().sum())

# чувствительность: считаем только измеренные часы, без интерполированных
measured = hourly["pm25"].where(~hourly["pm25_interpolated"])
measured_hours = measured.groupby(features.local_dates(hourly["time_utc"])).count()
full_dates = daily_raw["date"]
days_measured = (measured_hours[full_dates] >= features.MIN_HOURS_PER_DAY).sum()
print("суток с >= 18 часов только измеренных:", days_measured)
print("потеряли бы суток без интерполированных часов:", with_mean - days_measured)

часов в местных сутках -> число дат:
count
2        1
16       1
24    2043
25       1
Name: count, dtype: int64
           date  hours  pm25_hours  pm25_mean  temp_c  humidity  pressure  \
1420 2024-02-29     25          25      79.12  -6.924      78.2  1030.848   

      wind_speed  precip  
1420       1.858     0.0  
полных суток: 2044
первая дата: 2020-04-10 | последняя: 2025-11-13
pm25_hours
0         295
1-17      215
18-23      82
24       1451
25          1
Name: count, dtype: int64
суток с pm25_mean (>= 18 часов): 1534 | без: 510
суток с >= 18 часов только измеренных: 1496
потеряли бы суток без интерполированных часов: 38


## Шаг 7. Суточные признаки: норма ВОЗ, штиль, вчерашний PM2.5

- `who_exceed` — суточное среднее PM2.5 строго больше 15 µg/m³ (суточная норма ВОЗ 2021). Если среднего нет, значение `<NA>`, а не `False`: «нет данных» не равно «норма не превышена», иначе доля дней с превышением была бы занижена. Поэтому тип `boolean`.
- `is_calm` — штиль по **среднему за сутки** ветру (тот же порог: строго меньше 1 м/с).
- `pm25_lag1` — суточное среднее PM2.5 предыдущих календарных суток (для модели «завтра по сегодня» и наивного прогноза). Утечки нет: признак суток D берёт только сутки D−1. Перед сдвигом проверяется, что даты идут подряд; если вчерашнего среднего нет, лаг пустой — ничего не заполняем. Лаг считаем до любых `dropna`.
- `wind_dir` в суточную таблицу не берём: среднее углов неверно (350° и 10° дают 180°, хотя ветер почти северный). `cams_pm25` тоже нет: гипотеза H4 проверяется на часах.

In [8]:
daily = features.make_daily(hourly)
print(daily.head())

print(daily["who_exceed"].value_counts(dropna=False))

calm_days = daily["is_calm"].sum()
calm_heating = (daily["is_calm"] & daily["heating_season"]).sum()
print("суток со штилем:", calm_days, "| из них в отопительный сезон:", calm_heating)
print(daily["wind_speed"].describe())

print(daily["season"].value_counts())
print("суток в отопительный сезон:", daily["heating_season"].sum())

print("NaN в pm25_lag1:", daily["pm25_lag1"].isna().sum())
no_lag = daily["pm25_mean"].notna() & daily["pm25_lag1"].isna()
print("суток с pm25_mean, но без лага:", no_lag.sum())

# сутки для модели (6.1): есть и среднее, и вчерашнее среднее
for_model = daily[daily["pm25_mean"].notna() & daily["pm25_lag1"].notna()]
split_date = pd.Timestamp("2025-07-01")
print("суток с pm25_mean и pm25_lag1:", len(for_model))
print("  до 2025-06-30 включительно:", (for_model["date"] < split_date).sum())
print("  с 2025-07-01:", (for_model["date"] >= split_date).sum())

        date  pm25_mean  pm25_hours  pm25_lag1  who_exceed     temp_c  \
0 2020-04-10  10.791667          24        NaN       False   0.187500   
1 2020-04-11  37.958333          24  10.791667        True   7.516667   
2 2020-04-12  21.416667          24  37.958333        True  15.395833   
3 2020-04-13  10.916667          24  21.416667       False  12.691667   
4 2020-04-14  11.500000          24  10.916667       False  11.404167   

    humidity  precip     pressure  wind_speed  dayofweek  month  is_weekend  \
0  75.375000     0.0  1031.737500    2.002917          4      4       False   
1  61.791667     0.0  1019.654167    2.025833          5      4        True   
2  54.250000     0.0  1014.083333    2.029167          6      4        True   
3  81.625000    13.0  1015.704167    1.643333          0      4       False   
4  83.916667    12.3  1019.262500    1.256667          1      4       False   

   season  heating_season  is_calm  
0  spring            True    False  
1  spring   

## Шаг 8. Проверки и запись `daily.csv`

Контракт `check_daily` вызываем до записи и после чтения. Дата пишется как `YYYY-MM-DD` (без часового пояса), `who_exceed` при чтении получает тип `boolean`, поэтому `<NA>` сохраняется. Файл перезаписывается при каждом запуске.

In [9]:
features.check_daily(daily)
features.save_daily_csv(daily, DAILY_PATH)

daily_read = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily_read)
pd.testing.assert_frame_equal(daily, daily_read, check_dtype=False)

print("daily:", daily.shape, "| после чтения:", daily_read.shape)
print(daily_read["who_exceed"].dtype)

with open(DAILY_PATH, encoding="utf-8") as file:
    print(file.readline().strip())
    print(file.readline().strip())

daily: (2044, 16) | после чтения: (2044, 16)
boolean
date,pm25_mean,pm25_hours,pm25_lag1,who_exceed,temp_c,humidity,precip,pressure,wind_speed,dayofweek,month,is_weekend,season,heating_season,is_calm
2020-04-10,10.791666666666666,24,,False,0.18749999999999978,75.375,0.0,1031.7375,2.0029166666666662,4,4,False,spring,True,False


## Итог

Получили две таблицы (числа — в выводе шагов 3–8):
- `hourly.csv` — одна строка = один час на сетке станции: PM2.5, погода, CAMS, признаки по местному времени (вывод шагов 3–5);
- `daily.csv` — одна строка = одни полные местные сутки Алматы (вывод шагов 6–8).

Решения: осадки сдвинуты на час, чтобы совпасть с часом PM2.5 (ADR-014); CAMS до своего начала пустой и не заполняется; берём только полные сутки; среднее PM2.5 — при не менее 18 часов со значением, интерполированные часы считаются (цена — в выводе шага 6); `who_exceed` без данных — `<NA>`; `pm25_lag1` — по вчерашним календарным суткам.

Дальше:
- 4.1 и 5.1 берут `daily.csv` и `hourly.csv`;
- 5.2 (H4) берёт `hourly.csv`, где `cams_pm25` и `pm25` непустые одновременно;
- 6.1 использует `pm25_lag1`: разбиение train/test по времени, строки с пустыми значениями удалять только после построения лага; число суток для модели — в выводе шага 7.